# Product Funnel Analysis & A/B Hypothesis Prioritization

Анализ пользовательской воронки ride-hailing приложения на данных о **101 500 сессиях** и **24 224 пользователях**.

**Цель:** найти основные точки потерь, выявить проблемные сегменты и сформировать гипотезы для A/B-тестирования.

> Ноутбук подготовлен как portfolio case study. Исходный датасет не включён в репозиторий; для воспроизведения нужен файл `new_dataframe.xlsx`.

## 1. Импорт и загрузка данных

Основные инструменты: `pandas` для анализа и `matplotlib` для визуализации.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

df = pd.read_excel('new_dataframe.xlsx')
print('shape:', df.shape)
display(df.head())

## 2. Проверка качества данных

Проверяем типы, пропуски, полные совпадения строк и число уникальных пользователей.

In [ ]:
df.info()

quality = pd.DataFrame({
    'missing': df.isna().sum(),
    'unique': df.nunique(dropna=False)
})
display(quality)

print('duplicated rows:', df.duplicated().sum())
print('unique users:', df['user_id'].nunique())

### Вывод по пропускам

`surge` и `distance` отсутствуют ровно в тех сессиях, где `price_seen = 0`. Это **структурные пропуски**, а не случайно потерянные значения, поэтому их не следует заполнять средним или модой.

In [ ]:
missing_check = pd.DataFrame({
    'price_seen_0': [int((df['price_seen'] == 0).sum())],
    'distance_missing': [int(df['distance'].isna().sum())],
    'surge_missing': [int(df['surge'].isna().sum())],
    'distance_exact_match': [
        bool((df['distance'].isna() == (df['price_seen'] == 0)).all())
    ],
    'surge_exact_match': [
        bool((df['surge'].isna() == (df['price_seen'] == 0)).all())
    ]
})
display(missing_check)

### Логическая целостность событий

События образуют последовательную воронку:

`app_opened → price_seen → order_made → ride_completed / user_cancelled`

В данных нет заказов без просмотра цены, завершённых поездок без заказа или одновременно завершённых и отменённых поездок.

In [ ]:
checks = {
    'order_without_price': ((df['order_made'] == 1) & (df['price_seen'] == 0)).sum(),
    'completed_without_order': ((df['ride_completed'] == 1) & (df['order_made'] == 0)).sum(),
    'cancelled_without_order': ((df['user_cancelled'] == 1) & (df['order_made'] == 0)).sum(),
    'completed_and_cancelled': ((df['ride_completed'] == 1) & (df['user_cancelled'] == 1)).sum(),
}
display(pd.Series(checks, name='count'))

## 3. Базовая продуктовая воронка

Ключевые конверсии:

- `App opened → Price seen`: **90,08%**
- `Price seen → Order made`: **81,19%**
- `Order made → Ride completed`: **84,82%**
- `App opened → Ride completed`: **62,04%**

Главный провал находится после просмотра цены: около **18,8%** сессий, дошедших до цены, не переходят к заказу.

In [ ]:
funnel = pd.Series({
    'App opened': df['app_opened'].sum(),
    'Price seen': df['price_seen'].sum(),
    'Order made': df['order_made'].sum(),
    'Ride completed': df['ride_completed'].sum()
})

display(funnel)

conversion = pd.Series({
    'open_to_price': df['price_seen'].sum() / df['app_opened'].sum(),
    'price_to_order': df['order_made'].sum() / df['price_seen'].sum(),
    'order_to_complete': df['ride_completed'].sum() / df['order_made'].sum(),
    'open_to_complete': df['ride_completed'].sum() / df['app_opened'].sum(),
})
display(conversion.to_frame('conversion'))

ax = (funnel / funnel.iloc[0] * 100).plot(kind='bar', figsize=(8, 4))
ax.set_ylabel('% от открытий приложения')
ax.set_title('Product funnel')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Surge и конверсия в заказ

Среди сессий, дошедших до просмотра цены:

- без surge: **85,91%**
- с surge: **72,28%**
- разница: **13,62 п.п.**

Это сильная связь, но не причинный эффект: surge возникает в условиях повышенного спроса и может быть связан с другими факторами.

In [ ]:
surge_analysis = (
    df[df['price_seen'] == 1]
    .groupby('surge')
    .agg(
        sessions=('user_id', 'size'),
        orders=('order_made', 'sum'),
        order_rate=('order_made', 'mean')
    )
)
display(surge_analysis)

## 5. Время суток и взаимодействие с surge

Два выраженных периода снижения конверсии:

- **07:00–09:00**
- **19:00–21:00**

В эти же часы доля surge резко возрастает. Однако падение конверсии сохраняется даже при сравнении сессий с одинаковым статусом surge, поэтому временной эффект не сводится только к повышающему коэффициенту.

In [ ]:
hour_analysis = (
    df[df['price_seen'] == 1]
    .groupby('hour')
    .agg(
        sessions=('user_id', 'size'),
        order_rate=('order_made', 'mean'),
        surge_rate=('surge', lambda x: (x == 'surge').mean())
    )
)
display(hour_analysis)

hour_surge = (
    df[df['price_seen'] == 1]
    .groupby(['hour', 'surge'])['order_made']
    .mean()
    .unstack()
)
display(hour_surge)

hour_surge.plot(figsize=(10, 4), marker='o')
plt.ylabel('Price seen → Order made')
plt.title('Конверсия по часу и surge')
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

## 6. Класс поездки

На этапе `Price seen → Order made` различия между классами невелики. Существеннее различается поведение **после оформления заказа**:

- business: отмены **11,60%**
- comfort: **13,21%**
- economy: **17,81%**

`Economy` интересен дополнительно из-за большого объёма заказов.

In [ ]:
class_funnel = (
    df.groupby('order_class')
    .agg(
        sessions=('user_id', 'size'),
        price_seen=('price_seen', 'sum'),
        orders=('order_made', 'sum'),
        completed=('ride_completed', 'sum'),
        cancelled=('user_cancelled', 'sum')
    )
)
class_funnel['price_to_order'] = class_funnel['orders'] / class_funnel['price_seen']
class_funnel['order_to_cancel'] = class_funnel['cancelled'] / class_funnel['orders']
class_funnel['open_to_complete'] = class_funnel['completed'] / class_funnel['sessions']
display(class_funnel)

## 7. Android vs iOS

В сессиях Android наблюдаются:

- более низкая конверсия после просмотра цены;
- более высокая доля отмен после оформления заказа.

Эффект сохраняется при отдельных разрезах по классу, RFM, возрасту, дистанции и расположению заказа.

Важно: `os` не является постоянной характеристикой пользователя. Значительная часть `user_id` встречается на обеих ОС, поэтому корректнее говорить о **сессиях на Android/iOS**, а не о двух независимых группах пользователей.

In [ ]:
os_funnel = (
    df.groupby('os')
    .agg(
        sessions=('user_id', 'size'),
        price_seen=('price_seen', 'sum'),
        orders=('order_made', 'sum'),
        completed=('ride_completed', 'sum'),
        cancelled=('user_cancelled', 'sum')
    )
)
os_funnel['price_to_order'] = os_funnel['orders'] / os_funnel['price_seen']
os_funnel['order_to_cancel'] = os_funnel['cancelled'] / os_funnel['orders']
display(os_funnel)

os_per_user = df.groupby('user_id')['os'].nunique()
print('users with both OS:', (os_per_user > 1).sum())
print('share:', (os_per_user > 1).mean())

### Проверка на тех же пользователях

Даже среди пользователей, встречающихся на обеих ОС, разрыв сохраняется. Это снижает вероятность простого объяснения через разный состав пользователей, хотя причинность всё ещё не доказана.

In [ ]:
both_os_users = os_per_user[os_per_user > 1].index

same_users = (
    df[df['user_id'].isin(both_os_users)]
    .groupby('os')
    .agg(
        price_seen=('price_seen', 'sum'),
        orders=('order_made', 'sum'),
        completed=('ride_completed', 'sum'),
        cancelled=('user_cancelled', 'sum')
    )
)
same_users['price_to_order'] = same_users['orders'] / same_users['price_seen']
same_users['order_to_cancel'] = same_users['cancelled'] / same_users['orders']
display(same_users)

## 8. Центр города и дистанция

Заказы вне центра имеют более слабую воронку:

- отмены вне центра: **18,93%**
- из центра: **12,67%**

Для Android особенно проблемным оказался сегмент:

**Android × заказ вне центра × дистанция до ~4,3 км**

Cancellation rate в нём достигает примерно **24–25%**.

In [ ]:
df['distance_group'] = pd.qcut(df['distance'], q=4, duplicates='drop')

city = (
    df.groupby('city_center_order')
    .agg(
        price_seen=('price_seen', 'sum'),
        orders=('order_made', 'sum'),
        cancelled=('user_cancelled', 'sum')
    )
)
city['price_to_order'] = city['orders'] / city['price_seen']
city['order_to_cancel'] = city['cancelled'] / city['orders']
display(city)

interaction = (
    df[df['order_made'] == 1]
    .groupby(['os', 'city_center_order', 'distance_group'], observed=True)
    .agg(
        orders=('user_id', 'size'),
        cancelled=('user_cancelled', 'sum')
    )
)
interaction['cancel_rate'] = interaction['cancelled'] / interaction['orders']
display(interaction.sort_values('cancel_rate', ascending=False))

## 9. Проблемный сегмент × часы пик × surge

Для сегмента `Android × вне центра × distance ≤ 4,28 км`:

| Условия | Price seen → Order made |
|---|---:|
| вне пика, no surge | 80,60% |
| вне пика, surge | 74,38% |
| пик, no surge | 71,60% |
| пик, surge | **58,42%** |

Связь surge с падением конверсии особенно сильна именно в часы высокого спроса.

In [ ]:
df['peak_hour'] = df['hour'].isin([7, 8, 9, 19, 20, 21])

problem = df[
    (df['os'] == 'Android') &
    (df['city_center_order'] == 0) &
    (df['distance'] <= 4.28) &
    (df['price_seen'] == 1)
]

problem_peak_surge = (
    problem.groupby(['peak_hour', 'surge'])
    .agg(
        sessions=('user_id', 'size'),
        orders=('order_made', 'sum'),
        order_rate=('order_made', 'mean')
    )
)
display(problem_peak_surge)

## 10. Поведенческая сегментация пользователей

Переходим с уровня сессий на уровень пользователя и считаем индивидуальную историю прохождения воронки.

Дополнительная проверка показала, что cancellation rate почти не меняется с ростом количества заказов пользователя: около **15%** в группах от 1 до 5+ заказов. Значит, более активные пользователи чаще имеют хотя бы одну отмену прежде всего из-за большей экспозиции, а не из-за существенно большей склонности отменять.

In [ ]:
user_funnel = (
    df.groupby('user_id')
    .agg(
        app_opened=('app_opened', 'sum'),
        price_seen=('price_seen', 'sum'),
        orders=('order_made', 'sum'),
        completed=('ride_completed', 'sum'),
        cancelled=('user_cancelled', 'sum')
    )
)

user_funnel['cancel_rate'] = user_funnel['cancelled'] / user_funnel['orders']

user_funnel['orders_group'] = pd.cut(
    user_funnel['orders'],
    bins=[0, 1, 2, 4, float('inf')],
    labels=['1 заказ', '2 заказа', '3-4 заказа', '5+ заказов']
)

activity = (
    user_funnel[user_funnel['orders'] > 0]
    .groupby('orders_group', observed=True)
    .agg(
        users=('orders', 'size'),
        total_orders=('orders', 'sum'),
        total_cancelled=('cancelled', 'sum')
    )
)
activity['overall_cancel_rate'] = activity['total_cancelled'] / activity['total_orders']
display(activity)

## 11. SMART-гипотезы

На основе EDA сформированы семь гипотез:

1. Изменить коммуникацию повышенной стоимости при surge.
2. Упростить Android-экран оформления заказа.
3. Сделать ETA заметнее после оформления заказа в Android.
4. Добавить подтверждение точки подачи для заказов вне центра.
5. Точечно изменить сценарий для Android + вне центра + короткая поездка.
6. Адаптировать экран после расчёта цены в часы высокого спроса.
7. Проверить ограничение surge для наиболее проблемного сегмента в часы пик.

Каждая гипотеза должна проверяться отдельным экспериментом с заранее определённой основной метрикой и guardrail-метриками.

## 12. ICE-приоритизация

ICE = `Impact × Confidence × Ease`

Рабочая приоритизация:

| № | Impact | Confidence | Ease | ICE |
|---:|---:|---:|---:|---:|
| 1 | 8 | 7 | 7 | 392 |
| 2 | 9 | 8 | 7 | **504** |
| 3 | 8 | 6 | 6 | 288 |
| 4 | 8 | 6 | 7 | 336 |
| 5 | 9 | 7 | 5 | 315 |
| 6 | 7 | 6 | 5 | 210 |
| 7 | 10 | 9 | 5 | **450** |

Гипотеза №7 имеет высокий потенциальный эффект, но вмешивается в ценообразование. Перед реальным тестом необходимо добавить финансовые guardrail-метрики: выручку на сессию, маржу и число завершённых поездок.

## 13. Ограничения

- Анализ описательный; статистическая значимость различий здесь не проверялась.
- Данные наблюдательные, поэтому найденные связи не являются причинными.
- В данных нет фактической цены, выручки, маржи, ETA, числа доступных машин и дорожной ситуации.
- Сессии одного пользователя зависимы; будущую A/B-рандомизацию следует проводить по `user_id`.
- Однофакторные разрезы не заменяют многомерную модель.
- Сегменты найдены в ходе EDA и требуют подтверждения на новых данных или в эксперименте.

## 14. Следующий этап

Чтобы превратить кейс в полноценный A/B-проект:

1. выбрать одну гипотезу;
2. определить primary metric и guardrails;
3. сформулировать H0/H1;
4. задать MDE;
5. рассчитать sample size;
6. рандомизировать по `user_id`;
7. провести тест;
8. оценить эффект и доверительный интервал;
9. принять решение с учётом продуктовых и финансовых метрик.